# Pipeline Test

Run the MIMIC-FHIR pipeline inside Docker, then query the EDA table from DuckDB.

This notebook assumes:
- Docker Desktop is running
- data is mounted at `data/warehouse/mimic_fhir.duckdb`

## Pipeline services

| Service | Command | Purpose |
|---|---|---|
| `ingest` | `docker compose run --rm ingest` | Load raw NDJSON into bronze (run once) |
| `transform_vitals_eda` | `docker compose run --rm transform_vitals_eda` | Flatten vitals + build EDA models |

In [1]:
import subprocess
from pathlib import Path

import duckdb
import pandas as pd

project_root = Path.cwd().parent
db_path = project_root / "data" / "warehouse" / "mimic_fhir.duckdb"

print(f"Project root: {project_root}")
print(f"DuckDB path: {db_path}")


def run_cmd(cmd, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        cwd=project_root,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

Project root: /Users/kasra/Projects/MIMIC FHIR
DuckDB path: /Users/kasra/Projects/MIMIC FHIR/data/warehouse/mimic_fhir.duckdb


In [ ]:
# Run the transform pipeline (assumes bronze data already ingested).
# To re-ingest bronze: run_cmd(["docker", "compose", "run", "--rm", "ingest"])
run_cmd(["docker", "compose", "run", "--rm", "transform_vitals_eda"])

In [ ]:
# Read from DuckDB in read-only mode.
con = duckdb.connect(str(db_path), read_only=True)

checks = {
    "silver_rows": "SELECT COUNT(*) AS n FROM silver.obs_vitals",
    "wide_rows": "SELECT COUNT(*) AS n FROM intermediate.vitals_wide",
    "eda_rows": "SELECT COUNT(*) AS n FROM marts.vitals_eda",
}

for name, query in checks.items():
    n = con.execute(query).fetchone()[0]
    print(f"{name}: {n:,}")

eda_preview = con.execute(
    """
    SELECT *
    FROM marts.vitals_eda
    ORDER BY encounter_id, effective_datetime
    LIMIT 200
    """
).df()

eda_preview.head(20)

In [ ]:
# Optional cleanup when finished.
con.close()